# 第2回 宿題（発展）: 用語集を LoRA で学習させ、「学習させる」と「検索して渡す」を比べる

講座の用語集（58 語）で `Qwen/Qwen3.5-0.8B` を LoRA 学習し、同じ 116 問で次の条件を比べます。

| ラベル | 中身 |
| --- | --- |
| `base` | 素のモデル＋system プロンプト |
| `base_rag` | `base` ＋検索結果を入力に入れる |
| `ft_A` / `ft_B` | 学習後のモデル＋system プロンプト |
| `ft_A_rag` / `ft_B_rag` | 学習後のモデル＋検索結果 |

検索だけの条件（`search`）は GPU が要らないので、ローカルで `uv run python -m scripts.glossary_eval search` を実行します。

データの版は 2 つあります。

- **A（配布そのまま）**: 116 件、10 エポック（150 ステップ）。system なし、断る手本なし
- **B（改良版）**: 510 件、3 エポック（192 ステップ）。system あり、言い回しの追加、別名の手本、断る手本あり

**実行のしかた**

1. **ランタイム → ランタイムのタイプを変更** で G4 を選びます。**A と B は同じ GPU で回します**（`base` は A の回でしか測らないので、GPU が違うと B との比較に GPU の差が混ざります）
2. `VARIANT = "A"` のまま上から順に実行し、最後のセルの zip を保存します
3. **ランタイム → ランタイムを接続解除して削除** を実行します（LoRA はモデルの層に差し込まれるので、同じセッションで続けて学習できません）
4. `VARIANT` を `"B"` に変え、もう一度 G4 に接続して上から順に実行し、zip を保存して、ランタイムを削除します
5. 2 つの zip を `lectures/02/results/colab/` に展開します

**所要時間の目安**（元の演習の T4 の実測から計算した値です。G4 での実測はありません）

| 回 | 学習 | 生成 | 合計（T4） |
| --- | --- | --- | --- |
| A（4 ラベル × 116 問） | 5〜18 分 | 26〜95 分 | 31〜113 分 |
| B（2 ラベル × 116 問） | 11〜23 分 | 13〜47 分 | 24〜70 分 |

G4 なら、学習はおおむね 1/4〜1/8、生成は 1/2〜1/4 と見込んでいます。ただしこれは機種の性能差からの推測で、この実験で測った値ではありません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。

版: 1.0（2026-11-01）

## 0. リポジトリ、環境の確認、ライブラリの導入

**やること**: lec02 ブランチを `/content/repo` に clone し、検索（`app.tools`）と採点（`scripts.glossary_eval`）を読み込みます。続けて GPU を確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに、clone したコミットが表示されること。
- 2 つ目のセルに、G4 の GPU 名が表示されること。この名前は `result_{VARIANT}.json` に記録します。A と B で同じ名前になっていることを確かめてください。
- 3 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。

**うまくいかないとき**

- clone が失敗する: lec02 ブランチを push したか確かめます。
- ランタイムが同じままだと clone をやり直しません。push し直した内容を使うときは、ランタイムを削除してから実行します。
- pip の赤い `ERROR: pip's dependency resolver ...` は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この実験には影響しません。

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/content/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", "lec02", "https://github.com/deepkick/lecture-ai-engineering-2026", str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from app.tools import search_glossary
from scripts.glossary_eval import load_eval_set, score, summarize

REPO_COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", REPO_COMMIT)

In [ ]:
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
GPU_NAME = gpu_info.split(",")[0] if gpu_info else "none"
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → G4 を選んでください。")

In [ ]:
# 元のノートブックと同じ版（T4 で動作確認済み。G4 では未確認）。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
VERSIONS = {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]}
print(VERSIONS)

## 1. 設定

**やること**: データの版（`VARIANT`）とモデルの設定を決めます。`base_model`・`revision`・`seed`・`lr`・`lora_r`・`lora_alpha`・`max_seq_len` は元のノートブックと同じです。

- `VARIANT`: `"A"` か `"B"`。変えるときはランタイムを削除してから上から実行します。
- `RUN_BASELINES`: `base` と `base_rag` を評価するか。学習前のモデルは A でも B でも同じなので、A の回だけ `True` です。
- `max_new_tokens: 192`: 定義文は最長 95 トークンなので、答えが途中で切れない長さです。
- `USE_DRIVE = True`: Drive をマウントし、結果、チェックポイント、adapter を `MyDrive/aie2026/lec02_hw/{VARIANT}/` に保存します。ランタイムが消えても、ここから続けられます。最初に Drive へのアクセスの許可を求められます。
- `FORCE_RERUN = False`: 結果がすでにあるラベルは生成を飛ばし、学習済みなら adapter を読み込みます。`True` にすると、学習と生成をすべてやり直します。設定や評価セットを変えたときは `True` にしてください。

**見るところ**: 設定の JSON が表示されます。

**出力先**: 結果は `/content/lec02_hw/`（この中身をそのまま zip にします）。チェックポイントと adapter は大きいので結果と分けて、`USE_DRIVE` が `True` なら Drive、`False` なら `/content/lec02_work/` に置きます。

In [ ]:
import json, math, shutil, time
import pandas as pd

VARIANT = "A"            # "A" か "B"。変えるときはランタイムを削除してから上から実行する
EPOCHS = {"A": 10, "B": 3}
RUN_BASELINES = VARIANT == "A"   # 学習前のモデルは A でも B でも同じなので、A の回だけ評価する
RUN_FT_RAG = True
USE_DRIVE = True         # 結果・チェックポイント・adapter を Drive にも保存する
FORCE_RERUN = False      # True なら、結果があるラベルも生成し直し、学習もやり直す

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（条件を揃える）
    "VARIANT": VARIANT,
    "EPOCHS": EPOCHS,
    "RUN_BASELINES": RUN_BASELINES,
    "RUN_FT_RAG": RUN_FT_RAG,
    "USE_DRIVE": USE_DRIVE,
    "FORCE_RERUN": FORCE_RERUN,
}
OUT = Path("/content/lec02_hw"); OUT.mkdir(parents=True, exist_ok=True)   # 結果だけを置く（中身をそのまま zip にする）
DRIVE_DIR = Path("/content/drive/MyDrive/aie2026/lec02_hw") / VARIANT
STORE = DRIVE_DIR if USE_DRIVE else Path("/content/lec02_work")           # チェックポイントと adapter の置き場所
CKPT_DIR = STORE / f"ckpt_{VARIANT}"
ADAPTER_DIR = STORE / f"adapter_{VARIANT}"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    (DRIVE_DIR / "results").mkdir(parents=True, exist_ok=True)
    if not FORCE_RERUN:   # ランタイムが消えた後の再実行では、Drive の結果を戻して続きから進める
        for p in (DRIVE_DIR / "results").glob("*.json"):
            if not (OUT / p.name).exists():
                shutil.copy(p, OUT / p.name)
STORE.mkdir(parents=True, exist_ok=True)

def save_json(name: str, obj) -> None:
    '''結果を OUT に書き、USE_DRIVE なら Drive にもコピーする'''
    path = OUT / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    if USE_DRIVE:
        shutil.copy(path, DRIVE_DIR / "results" / name)

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("既にある結果:", sorted(p.name for p in OUT.glob("*.json")))

## 2. モデルの読み込み、生成と採点の準備

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、評価セットを読み、生成と採点の関数を用意します。

モデルへの入力は 3 通りです。

| 入力の形 | 中身 |
| --- | --- |
| 通常 | system ＋質問 |
| RAG | system ＋「用語集の検索結果:\n{検索結果}\n\n質問: {質問}」 |
| D（普通の質問） | system なしで質問だけ。RAG のラベルでも検索しない |

検索には、質問文をそのまま `search_glossary` に渡します。結果がよくない質問も含めて、仕組みのありのままの当たり方を見るためです。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: `116 問`（A・B・C・D の内訳）と、3 通りの入力の例が表示されます。

**出ても問題ない警告**: `You are sending unauthenticated requests to the HF Hub`（公開モデルなのでログインは不要です）、`... is falling back to its reference PyTorch implementation ...`（線形注意の層の実装の案内です）。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

In [ ]:
from collections import Counter

EVAL = load_eval_set(REPO / "lectures" / "02" / "homework" / "eval_set.json")
ITEMS = EVAL["items"]
SYSTEM = EVAL["system_prompt"]

def build_messages(item: dict, rag: bool) -> list[dict]:
    question = item["question"]
    if item["kind"] == "general":   # D は rag の値にかかわらず system なし・検索なし
        return [{"role": "user", "content": question}]
    user = f"用語集の検索結果:\n{search_glossary(question)}\n\n質問: {question}" if rag else question
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, messages: list[dict]) -> str:
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(len(ITEMS), "問", dict(Counter(i["category"] for i in ITEMS)), "| 評価セットの版:", EVAL["version"])
print("通常:", build_messages(ITEMS[0], rag=False)[-1]["content"])
print("RAG :", build_messages(ITEMS[0], rag=True)[-1]["content"][:120], "...")
print("D   :", build_messages(next(i for i in ITEMS if i["kind"] == "general"), rag=True))

**やること**: `evaluate` を定義します。116 問を生成して `glossary_eval.score` で採点し、`eval_{label}.json` に要約と行を保存します。

**採点のしかた**

- answer 型（A・B）: 断る表現に当たれば `over_refusal`。当たらなければ、必須語句で `correct` / `wrong`
- abstain 型（C）: 断る表現に当たれば `abstained`（正解）、当たらなければ `asserted`
- general 型（D）: 必須語句で `correct` / `wrong`
- 補助指標: answer 型では、答えと正解の定義文の文字単位 ROUGE-L（`rougeL`）も記録します

**見るところ**: このセルは定義だけなので、何も表示されません。

In [ ]:
SUMMARIES = {}

def has_lora(model) -> bool:
    '''LoRA の層が差し込まれているか。学習後は元の変数 model でも adapter 込みの出力になる'''
    return any("lora_" in name for name, _ in model.named_modules())

def show(summaries: dict) -> None:
    '''区分ごとの正解率と outcome の内訳を、ラベルを並べて表示する'''
    display(pd.DataFrame({label: {c: f"{g['ok']}/{g['n']}" for c, g in s["by_category"].items()} for label, s in summaries.items()}))
    display(pd.DataFrame({label: s["outcomes"] for label, s in summaries.items()}).fillna(0).astype(int))

def evaluate(model, label: str, rag: bool):
    '''116 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    path = OUT / f"eval_{label}.json"
    if path.exists() and not FORCE_RERUN:
        saved = json.loads(path.read_text(encoding="utf-8"))
        print(f"{path.name} があるので生成を飛ばします（commit {saved['summary']['commit']}）")
        SUMMARIES[label] = saved["summary"]
        return saved["summary"], saved["rows"]
    if label.startswith("base") and has_lora(model):
        raise RuntimeError("学習後のモデルで base を評価しようとしています。ランタイム → セッションを再起動して上から実行してください")
    rows = []
    t0 = time.time()
    for item in ITEMS:
        t1 = time.time()
        raw = generate(model, build_messages(item, rag))
        rows.append({"id": item["id"], "category": item["category"], "sub": item["sub"], "kind": item["kind"], "scope": item["scope"],
                     "question": item["question"], "raw": raw, **score(item, raw), "seconds": round(time.time() - t1, 2)})
    summary = {"label": label, "rag": rag, "seconds": round(time.time() - t0, 1), "gpu": GPU_NAME, "commit": REPO_COMMIT, **summarize(rows)}
    save_json(path.name, {"summary": summary, "rows": rows})
    SUMMARIES[label] = summary
    print(f"{label}: 正解率 {summary['rate']:.3f}（{summary['ok']}/{summary['n']}）| {summary['seconds']} 秒")
    return summary, rows

## 3. 学習前の評価（A の回だけ）

**やること**: `base` と `base_rag` をそれぞれ 116 問で評価します。

**見るところ**

- 区分ごとの正解数: A・B（答えられるか）、C（断れるか）、D（普通の質問）を分けて読みます。
- outcome の内訳: `asserted`（用語集にない語を断定した）と `over_refusal`（答えられるのに断った）のどちらが多いか。
- 出力の原文は `/content/lec02_hw/eval_base.json` にあります。必須語句で落ちていても、別の言葉で正しく説明していることがあるので、原文も読んでください。

B の回では何もしません。

In [ ]:
if RUN_BASELINES:
    evaluate(model, "base", rag=False)
    evaluate(model, "base_rag", rag=True)
    show(SUMMARIES)
else:
    print("RUN_BASELINES が False なので飛ばします")

## 4. LoRA 学習

**やること**: `glossary_sft_{VARIANT}.jsonl` を読み、元のノートブックと同じ LoRA の設定（`target_modules` も同じ）で学習します。

- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。端数のバッチも 1 ステップに数えるので、A は 15 × 10 = 150 ステップ、B は 64 × 3 = 192 ステップです。
- 学習率は cosine で減衰させ、`warmup_steps` は総ステップの約 10%（A 15、B 19）にします。
- チェックポイントは総ステップの約 1/4 ごと（A 38、B 48）に保存し、最新の 2 つを残します。
- **混合精度は使いません。** trl の `SFTConfig` は、`fp16=False` だけを渡すと `bf16=True` を既定にします。G4 と T4 で計算の精度が変わらないよう、`bf16=False` と `fp16=False` を両方明示しています。

シャッフルは `seed` に任せます。データファイルの行の順番は固定です。

**見るところ**

- 1 つ目のセル: 件数、総ステップ数、warmup、保存の間隔が上の数字と合うこと。
- 2 つ目のセル: `trainable%`、`Training Loss` の推移（5 ステップごと）、学習時間・ステップ数・最終 loss。ステップ数が予定と違うときは、警告が出ます。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...`

**うまくいかないとき**: 下の 4b を読んでください。

In [ ]:
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

train_rows = [json.loads(line) for line in (REPO / "lectures" / "02" / "results" / f"glossary_sft_{VARIANT}.jsonl").read_text(encoding="utf-8").splitlines()]
train_ds = Dataset.from_list(train_rows)

BATCH_SIZE, GRAD_ACCUM = 4, 2
STEPS_PER_EPOCH = math.ceil(math.ceil(len(train_rows) / BATCH_SIZE) / GRAD_ACCUM)   # 端数のバッチも 1 ステップに数える
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS[VARIANT]
WARMUP_STEPS = round(TOTAL_STEPS * 0.1)
SAVE_STEPS = round(TOTAL_STEPS / 4)
print(f"VARIANT={VARIANT} | {len(train_rows)} 件 × {EPOCHS[VARIANT]} エポック | 総ステップ {TOTAL_STEPS} | warmup {WARMUP_STEPS} | 保存 {SAVE_STEPS} ステップごと")
print("学習例の 1 件:", json.dumps(train_rows[0], ensure_ascii=False)[:200])

In [ ]:
assert not has_lora(model), "LoRA がすでに差し込まれています。ランタイム → セッションを再起動して上から実行してください"
TRAIN_JSON = OUT / f"train_{VARIANT}.json"

if TRAIN_JSON.exists() and ADAPTER_DIR.exists() and not FORCE_RERUN:
    print(f"{TRAIN_JSON.name} と adapter があるので学習を飛ばします")
    train_info = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
    tuned = PeftModel.from_pretrained(model, ADAPTER_DIR)
else:
    model.config.use_cache = False
    # Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
    lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    # bf16 を明示しないと trl の既定で True になる。重みも計算も 32bit のままにする
    sft = SFTConfig(output_dir=str(CKPT_DIR), num_train_epochs=EPOCHS[VARIANT], per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
                    learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS, logging_steps=5, save_steps=SAVE_STEPS, save_total_limit=2,
                    fp16=False, bf16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
    trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
    trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
    t0 = time.time()
    train_output = trainer.train()  # 途中で止まった場合の再開は 4b を参照
    train_seconds = round(time.time() - t0, 1)

    trainer.model.save_pretrained(ADAPTER_DIR)
    history = trainer.state.log_history
    save_json(f"log_{VARIANT}.json", history)
    trainable, all_params = trainer.model.get_nb_trainable_parameters()
    train_info = {
        "seconds": train_seconds,
        "steps": trainer.state.global_step,
        "expected_steps": TOTAL_STEPS,
        "epoch": trainer.state.epoch,
        "final_loss": [e["loss"] for e in history if "loss" in e][-1],
        "mean_loss": train_output.training_loss,
        "trainable_params": trainable,
        "all_params": all_params,
        "trainable_pct": 100 * trainable / all_params,
        "n_train": len(train_rows),
        "warmup_steps": WARMUP_STEPS,
        "save_steps": SAVE_STEPS,
        "bf16": trainer.args.bf16,
        "fp16": trainer.args.fp16,
    }
    save_json(TRAIN_JSON.name, train_info)
    tuned = trainer.model

tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）
print(f"学習時間: {train_info['seconds']} 秒 | ステップ数: {train_info['steps']} | エポック: {train_info['epoch']:.1f} | "
      f"最終 loss: {train_info['final_loss']:.4f} | trainable%: {train_info['trainable_pct']:.2f}")
if train_info["steps"] != TOTAL_STEPS:
    print(f"警告: ステップ数が予定（{TOTAL_STEPS}）と違います")

### 4b. 学習が途中で止まった場合

チェックポイントは総ステップの約 1/4 ごと（A は 38、B は 48 ステップごと）に、`ckpt_{VARIANT}/` へ保存しています。`USE_DRIVE = True` なら Drive にあるので、ランタイムが消えても残ります。

1. **ランタイム → セッションを再起動** します。ランタイムが消えた場合は、もう一度 G4 に接続します（A と B で同じ GPU にします）。
2. セクション 0〜3 を上から実行します。`FORCE_RERUN = False` のままにしてください。結果があるラベルは生成を飛ばします。
3. 上の学習セルの `train_output = trainer.train()` を `train_output = trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから総ステップまで続きを学習します。

`seconds` は再開後の分だけになります。answer.md に書くときは、途中で再開したことも書いてください。

**学習の後で止まった場合**: `train_{VARIANT}.json` と adapter があれば、上から実行するだけで学習を飛ばして adapter を読み込み、残りのラベルだけ生成します。

**Drive の容量**: 学習が終わったら `ckpt_{VARIANT}/` は消してかまいません。

## 5. 学習後の評価

**やること**: 学習後のモデル `tuned` を同じ 116 問、同じ生成設定で評価します。`RUN_FT_RAG` が `True` なら、検索結果を入れた形でも評価します。

**見るところ**

- 区分ごとの正解数: A（学習と同じ聞き方）に比べて、B1〜B4（学習していない聞き方）でどれだけ落ちるか。
- C（用語集にない語）: `abstained` と `asserted` のどちらが多いか。
- D（普通の質問）: 学習で壊れていないか。
- 手作り（B1〜B3、C1〜C3）と機械的な拡張（B4、C4）で傾向が一致するか。

In [ ]:
evaluate(tuned, f"ft_{VARIANT}", rag=False)
if RUN_FT_RAG:
    evaluate(tuned, f"ft_{VARIANT}_rag", rag=True)
show(SUMMARIES)

## 6. 結果の保存

**やること**: 設定、ライブラリの版、GPU 名、学習の数字、各ラベルの要約を `result_{VARIANT}.json` にまとめ、`/content/lec02_hw/` の中身を `lec02_{VARIANT}_results.zip` にします。チェックポイントと adapter は別の場所にあるので、zip には入りません。

**見るところ**: 最後に zip の中身とダウンロードの方法が表示されます。**ランタイムを削除する前に zip を保存してください。** `USE_DRIVE = True` なら Drive にもコピーされます。

保存したら **ランタイム → ランタイムを接続解除して削除** を実行します。

In [ ]:
result = {
    "config": CONFIG,
    "versions": VERSIONS,
    "gpu": GPU_NAME,
    "repo_commit": REPO_COMMIT,
    "eval_set_version": EVAL["version"],
    "method": "LoRA（量子化なし）",
    "base_memory_gb": BASE_MEMORY_GB,
    "train": train_info,
    "summaries": SUMMARIES,
    "executed_training_myself": True,
}
save_json(f"result_{VARIANT}.json", result)
print(json.dumps({k: v for k, v in result.items() if k != "summaries"}, ensure_ascii=False, indent=2))
show(SUMMARIES)

zip_path = Path(shutil.make_archive(f"/content/lec02_{VARIANT}_results", "zip", OUT))
if USE_DRIVE:
    shutil.copy(zip_path, DRIVE_DIR / zip_path.name)
print("zip の中身:", sorted(p.name for p in OUT.iterdir()))
print(f"ダウンロード: 左のファイル一覧で {zip_path} を右クリック → ダウンロード。または次の 2 行を実行する")
print("  from google.colab import files")
print(f"  files.download('{zip_path}')")
if USE_DRIVE:
    print("Drive にも保存しました:", DRIVE_DIR / zip_path.name)